# V6 — 06: Eval Baselines (B1–B4)

Two eval rounds, each with 4 models × 2 episode counts = 8 jobs.

| Round | Models        | m=50 | m=100 |
|-------|---------------|------|-------|
| 1     | B1, B2, B3, B4 | ✓   | ✓     |
| 2     | B1, B2, B3, B4 | ✓   | ✓     |

Each run uses 4 GPUs (one per model), running m=50 and m=100 sequentially per GPU.

In [ ]:
import sys, os
from pathlib import Path

GPU_OFFSET = 0  # ← only line to change

sys.path.insert(0, str(Path(".").resolve()))
import common_v6 as v6

os.environ.setdefault("MUJOCO_GL", "osmesa")
os.environ.setdefault("PYOPENGL_PLATFORM", "osmesa")

TAGS = ["B1", "B2", "B3", "B4"]

# EVAL_JOBS: list of (tag, gpu_local, n_episodes)
# m=50 and m=100 are run sequentially on the same GPU (one job per GPU per round)
EVAL_JOBS_R1 = [(tag, i, 50)  for i, tag in enumerate(TAGS)]
EVAL_JOBS_R2 = [(tag, i, 100) for i, tag in enumerate(TAGS)]

In [ ]:
# Status check before launching
print("=== Training status ===")
v6.print_training_status(TAGS)
print()
print("=== Eval status ===")
v6.print_eval_status(TAGS)

In [ ]:
# Round 1: m=50
print("Launching Round 1 (m=50) ...")
procs_r1 = v6.launch_eval(EVAL_JOBS_R1, GPU_OFFSET)
print("Monitor:")
for tag in TAGS:
    log = v6.get_output_dir(tag) / "eval" / "eval.log"
    print(f"  tail -f {log}")

In [ ]:
# Round 2: m=100 (run after Round 1 finishes)
print("Launching Round 2 (m=100) ...")
procs_r2 = v6.launch_eval(EVAL_JOBS_R2, GPU_OFFSET)
print("Monitor:")
for tag in TAGS:
    log = v6.get_output_dir(tag) / "eval" / "eval.log"
    print(f"  tail -f {log}")

In [ ]:
# Post-processing: collect results and compute Wilson CIs
import json

results = {}
for tag in TAGS:
    m = v6.build_metrics_from_eval_info(tag)
    results[tag] = m
    v6.save_metrics(tag, m)

print(f"{'TAG':<8} {'LABEL':<30} {'SR':>6} {'CI_LO':>7} {'CI_HI':>7}")
print("-" * 65)
for tag in TAGS:
    m = results.get(tag, {})
    sr = m.get('sr')
    lo = m.get('sr_ci_lo')
    hi = m.get('sr_ci_hi')
    label = m.get('label', tag)
    sr_s  = f"{sr:.3f}" if sr is not None else " ─"
    lo_s  = f"{lo:.3f}" if lo is not None else " ─"
    hi_s  = f"{hi:.3f}" if hi is not None else " ─"
    print(f"{tag:<8} {label:<30} {sr_s:>6} {lo_s:>7} {hi_s:>7}")